In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import joblib

def train_gmm_personas():
    input_file = "cf_ml_features_clean.csv"
    
    print(f"[*] Loading clean data from '{input_file}'...")
    try:
        df = pd.read_csv(input_file)
    except FileNotFoundError:
        print("[-] Error: Clean feature file not found. Run DBSCAN first.")
        return

    features_to_drop = ['handle', 'total_submissions']
    if 'dbscan_label' in df.columns:
        features_to_drop.append('dbscan_label')
        
    X = df.drop(columns=features_to_drop)
    feature_names = X.columns
    
    print("[*] Scaling features...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print("[*] Training the GMM Persona Model (k=10)...")
    gmm = GaussianMixture(n_components=10, covariance_type='full', random_state=42)
    
    cluster_labels = gmm.fit_predict(X_scaled)
    df['Persona_Cluster'] = cluster_labels
    
    print("\n[*] Saving models for your backend server...")
    joblib.dump(scaler, 'persona_scaler.pkl')
    joblib.dump(gmm, 'persona_gmm_model.pkl')
    print("[+] Saved 'persona_scaler.pkl' and 'persona_gmm_model.pkl'")
    
    # ---------------------------------------------------------
    # THE FIX: CALCULATE GLOBAL MEANS FOR LIFT
    # ---------------------------------------------------------
    tag_columns = {
        'math_pref': 'Math', 
        'dp_pref': 'Dynamic Programming', 
        'graph_pref': 'Graphs/Trees', 
        'brute_pref': 'Brute Force/Impl',
        'greedy_pref': 'Greedy/Two Pointers',
        'binary_pref': 'Binary Search',
        'cons_pref': 'Constructive',
        'datastruct_pref': 'Data Structures'
    }
    
    # Calculate the average across ALL 1,500 users
    global_means = df[list(tag_columns.keys())].mean()
    
    print("\n" + "="*55)
    print("THE 10 CODER PERSONAS DISCOVERED BY AI:")
    print("="*55)
    
    cluster_means = df.groupby('Persona_Cluster')[feature_names].mean()
    
    for cluster_id in range(10):
        print(f"\n[ Archetype {cluster_id} ] - ({len(df[df['Persona_Cluster'] == cluster_id])} users)")
        
        profile = cluster_means.loc[cluster_id]
        
        print(f"  -> Accuracy:           {profile['accuracy']*100:.1f}%")
        print(f"  -> One-Shot Precision: {profile['one_shot_rate']*100:.1f}%")
        print(f"  -> TLE (Optimization): {profile['optimization_struggle']*100:.1f}%")
        print(f"  -> Abandonment Rate:   {profile['abandonment_rate']*100:.1f}%")
        print(f"  -> Tilt Speed:         {profile['tilt_speed_seconds']:.0f} seconds")
        print(f"  -> Avg Solved Rating:  {profile['avg_solved_rating']:.0f}")
        
        # Calculate RELATIVE Lift for each tag
        lifts = {}
        for col, human_name in tag_columns.items():
            # Prevent division by zero just in case
            if global_means[col] > 0:
                lift = profile[col] / global_means[col]
            else:
                lift = 0
            lifts[human_name] = lift
            
        # Find the tag with the highest multiplier relative to the global average
        best_domain = max(lifts, key=lifts.get)
        best_multiplier = lifts[best_domain]
        
        print(f"  -> Defining Domain:    {best_domain} ({best_multiplier:.2f}x average)")

if __name__ == "__main__":
    train_gmm_personas()

[*] Loading clean data from 'cf_ml_features_clean.csv'...
[*] Scaling features...
[*] Training the GMM Persona Model (k=10)...

[*] Saving models for your backend server...
[+] Saved 'persona_scaler.pkl' and 'persona_gmm_model.pkl'

THE 10 CODER PERSONAS DISCOVERED BY AI:

[ Archetype 0 ] - (85 users)
  -> Accuracy:           61.5%
  -> One-Shot Precision: 66.8%
  -> TLE (Optimization): 2.6%
  -> Abandonment Rate:   29.9%
  -> Tilt Speed:         256 seconds
  -> Avg Solved Rating:  1098
  -> Defining Domain:    Brute Force/Impl (1.13x average)

[ Archetype 1 ] - (179 users)
  -> Accuracy:           45.9%
  -> One-Shot Precision: 53.6%
  -> TLE (Optimization): 7.5%
  -> Abandonment Rate:   34.9%
  -> Tilt Speed:         255 seconds
  -> Avg Solved Rating:  1176
  -> Defining Domain:    Greedy/Two Pointers (1.08x average)

[ Archetype 2 ] - (103 users)
  -> Accuracy:           51.8%
  -> One-Shot Precision: 57.1%
  -> TLE (Optimization): 5.3%
  -> Abandonment Rate:   19.9%
  -> Tilt Spe